# MA3632 — Workshop 8: Decision Trees

This workshop accompanies Lecture 8. Part A implements Gini impurity, entropy, and
the greedy split search from scratch. Parts B-D fit, visualise, and prune trees using
scikit-learn. Parts E and F study feature importance measures and MDI bias.

Work through all parts in order. Take-home exercises are at the end.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.tree import (
    DecisionTreeClassifier, export_text, plot_tree
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)

# ── Load Digits dataset once ──────────────────────────────────────────────────
# 1797 samples, 64 pixel-intensity features (8x8 images), 10 classes (digits 0-9).
digits = load_digits()
X_dg, y_dg = digits.data, digits.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_dg, y_dg, test_size=0.3, random_state=0, stratify=y_dg
)

print(f"Digits: {X_dg.shape[0]} samples, {X_dg.shape[1]} features, {len(np.unique(y_dg))} classes")
print(f"Train / test: {len(y_tr)} / {len(y_te)}")
print(f"Class counts (train): {np.bincount(y_tr)}")

---
## Part A — Impurity measures and the greedy split from scratch

We implement Gini impurity and entropy, reproduce the lecture's numerical examples,
and build a single-level greedy best-split search.

### A1. Gini impurity and entropy

In [ ]:
def gini(y):
    """Gini impurity of a node with label vector y."""
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y, minlength=y.max() + 1)
    p = counts / n
    return 1.0 - np.sum(p ** 2)

def entropy(y):
    """Entropy (nats) of a node with label vector y."""
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y, minlength=y.max() + 1)
    p = counts[counts > 0] / n
    return -np.sum(p * np.log(p))

# Reproduce the lecture table: pure node, balanced binary, uniform over k classes
print("{:<35} {:>8} {:>8}".format("Distribution", "Gini", "H/ln2"))
print("-" * 53)

cases = [
    ("Pure node (all class 0)",          np.array([0, 0, 0, 0, 0])),
    ("Binary, balanced (2 classes)",     np.array([0, 0, 1, 1])),
    ("Ternary, balanced (3 classes)",    np.array([0, 0, 1, 1, 2, 2])),
    ("Uniform over 10 classes",          np.repeat(np.arange(10), 5)),
]
for label, y in cases:
    g = gini(y)
    h = entropy(y) / np.log(2)
    print(f"{label:<35} {g:>8.4f} {h:>8.4f}")

### A2. Impurity reduction (information gain)

In [ ]:
def impurity_reduction(y, y_left, y_right, criterion="gini"):
    """Weighted impurity reduction of a split."""
    fn = gini if criterion == "gini" else entropy
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    return fn(y) - (n_l / n) * fn(y_left) - (n_r / n) * fn(y_right)

# Worked example in the same spirit as the lecture's split-impurity calculation
# (Section 3, Example 3.5), using a different parent split to additionally
# contrast a partially-separating split against a perfectly-separating one:
# Parent: 5 class-0, 5 class-1.  Split A: {4,1} / {1,4}.  Split B: {5,0} / {0,5}.
y_parent = np.array([0,0,0,0,0,1,1,1,1,1])
y_A_left = np.array([0,0,0,0,1]); y_A_right = np.array([0,1,1,1,1])
y_B_left = np.array([0,0,0,0,0]); y_B_right = np.array([1,1,1,1,1])

print("Split A (impure both sides):")
print(f"  Gini reduction: {impurity_reduction(y_parent, y_A_left, y_A_right):.4f}")
print(f"  Entr reduction: {impurity_reduction(y_parent, y_A_left, y_A_right, 'entropy'):.4f}")
print("Split B (perfect separation):")
print(f"  Gini reduction: {impurity_reduction(y_parent, y_B_left, y_B_right):.4f}")
print(f"  Entr reduction: {impurity_reduction(y_parent, y_B_left, y_B_right, 'entropy'):.4f}")

### A3. Greedy best-split search on a single feature

In [ ]:
def best_split_1d(x, y):
    """Find the threshold on a single feature that maximises Gini reduction."""
    thresholds = np.unique(x)
    best_gain, best_t = -1.0, None
    for t in thresholds:
        mask = x <= t
        if mask.sum() == 0 or (~mask).sum() == 0:
            continue
        gain = impurity_reduction(y, y[mask], y[~mask])
        if gain > best_gain:
            best_gain, best_t = gain, t
    return best_t, best_gain

# Apply to pixel 27 (a central pixel) on a binary subset: digit 0 vs digit 1
mask_01 = np.isin(y_tr, [0, 1])
x_demo = X_tr[mask_01, 27]
y_demo = (y_tr[mask_01] == 1).astype(int)

t_star, gain_star = best_split_1d(x_demo, y_demo)
print(f"Best split on pixel 27 (digit 0 vs 1):")
print(f"  threshold = {t_star:.1f},  Gini reduction = {gain_star:.4f}")
print(f"  Left ({(x_demo <= t_star).sum()} pts): digit 0 fraction = "
      f"{(y_demo[x_demo <= t_star] == 0).mean():.3f}")
print(f"  Right ({(x_demo > t_star).sum()} pts): digit 1 fraction = "
      f"{(y_demo[x_demo > t_star] == 1).mean():.3f}")

**In-class exercise.** Repeat the best-split search above using entropy instead
of Gini impurity by passing `criterion='entropy'` to `impurity_reduction`. Do the
two criteria select the same threshold? In general, Gini and entropy tend to agree
on which split to make but can differ near ties — the main practical difference is
computational cost rather than predictive performance.

---
## Part B — Fitting and visualising a decision tree

We fit a depth-3 tree on the full Digits training set, print its text representation,
and plot the decision boundary on a two-feature projection.

In [ ]:
clf3 = DecisionTreeClassifier(max_depth=3, random_state=0)
clf3.fit(X_tr, y_tr)

print(export_text(clf3, feature_names=[f"px{i}" for i in range(64)]))

In [ ]:
# Plot the tree diagram (limited to depth 3 for readability)
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(clf3, feature_names=[f"px{i}" for i in range(64)],
          class_names=[str(d) for d in range(10)],
          filled=True, rounded=True, fontsize=7, ax=ax)
ax.set_title("Decision tree (max_depth=3) — Digits", fontsize=12)
plt.tight_layout(); plt.show()
print(f"Leaves: {clf3.get_n_leaves()}   Depth: {clf3.get_depth()}")
print(f"Train accuracy: {clf3.score(X_tr, y_tr):.4f}")
print(f"Test accuracy:  {clf3.score(X_te, y_te):.4f}")

In [ ]:
# 2D decision boundary on pixels 27 and 35 (two central pixels)
feat_a, feat_b = 27, 35
X_tr_2d = X_tr[:, [feat_a, feat_b]]
X_te_2d = X_te[:, [feat_a, feat_b]]

clf_2d = DecisionTreeClassifier(max_depth=4, random_state=0)
clf_2d.fit(X_tr_2d, y_tr)

h = 0.2
x1_min, x1_max = X_dg[:, feat_a].min() - 0.5, X_dg[:, feat_a].max() + 0.5
x2_min, x2_max = X_dg[:, feat_b].min() - 0.5, X_dg[:, feat_b].max() + 0.5
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h),
                     np.arange(x2_min, x2_max, h))
Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(xx, yy, Z, cmap="tab10", alpha=0.4)
sc = ax.scatter(X_tr_2d[:, 0], X_tr_2d[:, 1], c=y_tr,
                cmap="tab10", edgecolors="k", s=14, linewidths=0.3)
ax.set_xlabel(f"Pixel {feat_a}"); ax.set_ylabel(f"Pixel {feat_b}")
ax.set_title("Decision boundary (depth 4, 2 features) — Digits")
plt.colorbar(sc, ax=ax, label="Digit class")
plt.tight_layout(); plt.show()

**In-class exercise.** The decision boundaries are strictly axis-aligned rectangles.
Explain why this is an intrinsic property of binary decision trees, regardless of
depth. What class of geometric shapes can a depth-$d$ tree represent in two dimensions?

---
## Part C — Depth, bias-variance, and tree instability

We study how depth controls the bias-variance trade-off and demonstrate that
unpruned trees are highly sensitive to the training sample.

In [ ]:
# Training vs test error over depth 1-20
depths = list(range(1, 21))
train_err, test_err = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=0)
    clf.fit(X_tr, y_tr)
    train_err.append(1 - clf.score(X_tr, y_tr))
    test_err.append(1 - clf.score(X_te, y_te))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_err, "o-", label="Train error", color="steelblue")
plt.plot(depths, test_err,  "o-", label="Test error",  color="darkorange")
plt.xlabel("max_depth"); plt.ylabel("Misclassification rate")
plt.title("Depth vs error — Digits")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 5-fold CV with 1-SE rule
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_mean, cv_se = [], []
for d in depths:
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=d, random_state=0),
        X_tr, y_tr, cv=skf, scoring="accuracy"
    )
    cv_mean.append(1 - scores.mean())
    cv_se.append(scores.std() / np.sqrt(5))

cv_mean, cv_se = np.array(cv_mean), np.array(cv_se)
best_idx = cv_mean.argmin()
threshold = cv_mean[best_idx] + cv_se[best_idx]
ose_idx = np.where(cv_mean <= threshold)[0][0]

plt.figure(figsize=(9, 4))
plt.plot(depths, cv_mean, "o-", color="steelblue", label="CV error")
plt.fill_between(depths, cv_mean - cv_se, cv_mean + cv_se, alpha=0.2, color="steelblue")
plt.axhline(threshold, color="darkorange", ls="--", label="1-SE threshold")
plt.axvline(depths[ose_idx], color="green", ls=":", label=f"1-SE choice: depth={depths[ose_idx]}")
plt.xlabel("max_depth"); plt.ylabel("CV misclassification rate")
plt.title("5-fold CV with 1-SE rule — Digits")
plt.legend(); plt.tight_layout(); plt.show()
print(f"CV minimiser: depth = {depths[best_idx]}  (error = {cv_mean[best_idx]:.4f})")
print(f"1-SE choice:  depth = {depths[ose_idx]}  (error = {cv_mean[ose_idx]:.4f})")

In [ ]:
# Tree instability: two trees on bootstrap resamples
idx_a = rng.integers(0, len(X_tr), len(X_tr))
idx_b = rng.integers(0, len(X_tr), len(X_tr))

clf_a = DecisionTreeClassifier(max_depth=None, random_state=0)
clf_b = DecisionTreeClassifier(max_depth=None, random_state=1)
clf_a.fit(X_tr[idx_a], y_tr[idx_a])
clf_b.fit(X_tr[idx_b], y_tr[idx_b])

feat_a_root = clf_a.tree_.feature[0]
feat_b_root = clf_b.tree_.feature[0]
print(f"Tree A root split: pixel {feat_a_root}")
print(f"Tree B root split: pixel {feat_b_root}")
print(f"Same root feature: {feat_a_root == feat_b_root}")

agree = (clf_a.predict(X_te) == clf_b.predict(X_te)).mean()
print(f"Agreement on test predictions: {agree:.4f}")
print("Even modest bootstrap variation produces different structures and predictions.")

---
## Part D — Cost-complexity pruning

We grow the full unpruned tree, extract the pruning path, and select the best
$\alpha$ by cross-validation.

In [ ]:
# Grow the full tree
clf_full = DecisionTreeClassifier(random_state=0)
clf_full.fit(X_tr, y_tr)
print(f"Full tree: depth = {clf_full.get_depth()}, leaves = {clf_full.get_n_leaves()}")
print(f"Train error: {1 - clf_full.score(X_tr, y_tr):.4f}")
print(f"Test error:  {1 - clf_full.score(X_te, y_te):.4f}")

In [ ]:
# Extract the pruning path
path = clf_full.cost_complexity_pruning_path(X_tr, y_tr)
alphas, impurities = path.ccp_alphas, path.impurities

# Crossover values of alpha (where optimal subtree changes)
crossovers = np.where(np.diff(alphas) > 0)[0]
print(f"Pruning path: {len(alphas)} subtrees")
print(f"Alpha range: [{alphas[0]:.6f}, {alphas[-1]:.6f}]")
print(f"Number of crossover values: {len(crossovers)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(alphas[:-1], [DecisionTreeClassifier(ccp_alpha=a, random_state=0)
                             .fit(X_tr, y_tr).get_n_leaves()
                             for a in alphas[:-1]], "o-", ms=3, color="steelblue")
axes[0].set_xscale("log"); axes[0].set_xlabel("alpha (log scale)")
axes[0].set_ylabel("Number of leaves")
axes[0].set_title("Leaves vs alpha")

axes[1].plot(alphas, impurities, "o-", ms=3, color="darkorange")
axes[1].set_xscale("log"); axes[1].set_xlabel("alpha (log scale)")
axes[1].set_ylabel("Total impurity")
axes[1].set_title("Total impurity vs alpha")
plt.tight_layout(); plt.show()

In [ ]:
# Cross-validation to select best alpha
alpha_candidates = alphas[:-1][alphas[:-1] > 0]
# Use a coarser grid for speed
idx_grid = np.round(np.linspace(0, len(alpha_candidates)-1, 30)).astype(int)
alpha_grid = alpha_candidates[idx_grid]

cv_scores = []
for a in alpha_grid:
    scores = cross_val_score(
        DecisionTreeClassifier(ccp_alpha=a, random_state=0),
        X_tr, y_tr, cv=5, scoring="accuracy"
    )
    cv_scores.append(scores.mean())

best_alpha = alpha_grid[np.argmax(cv_scores)]
print(f"CV-optimal alpha: {best_alpha:.6f}")

clf_pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=0)
clf_pruned.fit(X_tr, y_tr)
print(f"Pruned tree: depth = {clf_pruned.get_depth()}, leaves = {clf_pruned.get_n_leaves()}")
print(f"Train error: {1 - clf_pruned.score(X_tr, y_tr):.4f}")
print(f"Test error:  {1 - clf_pruned.score(X_te, y_te):.4f}")
print(f"Full tree test error:   {1 - clf_full.score(X_te, y_te):.4f}")
print(f"Pruned tree test error: {1 - clf_pruned.score(X_te, y_te):.4f}")

**In-class exercise.** The pruning path above contains many subtrees. For three values
of alpha — one well below, one near, and one well above the CV-optimal alpha —
state the number of leaves and the test error. What pattern do you observe as alpha
increases from left to right along the path?

---
## Part E — MDI vs permutation importance

We compare mean decrease in impurity (MDI) to permutation importance on the
pruned tree from Part D, and examine their agreement on the top features.

In [ ]:
# MDI importance (built into the fitted tree)
mdi = clf_pruned.feature_importances_
feat_names = [f"px{i}" for i in range(64)]

# Permutation importance on test set
perm = permutation_importance(
    clf_pruned, X_te, y_te, n_repeats=20, random_state=0, scoring="accuracy"
)
perm_mean = perm.importances_mean

# Top-10 by each measure
top10_mdi  = np.argsort(mdi)[::-1][:10]
top10_perm = np.argsort(perm_mean)[::-1][:10]
overlap = len(set(top10_mdi) & set(top10_perm))
print(f"Top-10 overlap between MDI and permutation importance: {overlap}/10")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, title, colour in zip(
        axes,
        [mdi[top10_mdi], perm_mean[top10_perm]],
        ["MDI (top 10)", "Permutation importance (top 10)"],
        ["steelblue", "darkorange"]):
    idx = top10_mdi if "MDI" in title else top10_perm
    ax.bar(range(10), scores, color=colour)
    ax.set_xticks(range(10))
    ax.set_xticklabels([feat_names[i] for i in idx], rotation=45, ha="right", fontsize=8)
    ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# Visualise the most important pixel locations on the 8x8 grid
mdi_grid = mdi.reshape(8, 8)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(mdi_grid, cmap="hot")
axes[0].set_title("MDI by pixel location"); plt.colorbar(im0, ax=axes[0])

perm_grid = perm_mean.reshape(8, 8)
im1 = axes[1].imshow(perm_grid, cmap="hot")
axes[1].set_title("Permutation importance by pixel location"); plt.colorbar(im1, ax=axes[1])
plt.suptitle("Feature importance heatmaps — Digits (8x8 grid)", y=1.02)
plt.tight_layout(); plt.show()

The heatmaps show which pixel positions the tree relies on most. Central pixels
tend to receive higher importance than corner pixels, which are almost always zero
and carry little discriminative information across digit classes.

---
## Part F — MDI bias with high-cardinality features

MDI is known to overestimate the importance of features with many distinct values
(high cardinality), because such features offer more candidate thresholds and are
therefore more likely to be selected at each node by chance. We demonstrate this
with a synthetic dataset containing one genuinely informative feature and one
high-cardinality noise feature.

In [ ]:
# Synthetic: 5 informative features, 1 high-cardinality noise feature
# The noise feature has 100 unique float values; informative features are binary-ish.
n = 600
X_inf = rng.standard_normal((n, 5))
X_noise_hc = rng.uniform(0, 1, (n, 1)) * 100   # high cardinality, pure noise
X_noise_lc = rng.integers(0, 3, (n, 1)).astype(float)  # low cardinality, pure noise
X_bias = np.hstack([X_inf, X_noise_hc, X_noise_lc])
y_bias = (X_inf[:, 0] + X_inf[:, 1] > 0).astype(int)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bias, y_bias, test_size=0.3, random_state=0
)

clf_bias = DecisionTreeClassifier(max_depth=6, random_state=0)
clf_bias.fit(X_tr_b, y_tr_b)

mdi_bias = clf_bias.feature_importances_
feat_labels = [f"inf_{i}" for i in range(5)] + ["noise_HC", "noise_LC"]

perm_bias = permutation_importance(
    clf_bias, X_te_b, y_te_b, n_repeats=20, random_state=0
).importances_mean

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colours = ["steelblue"]*5 + ["firebrick", "darkorange"]
for ax, scores, title in zip(
        axes,
        [mdi_bias, perm_bias],
        ["MDI", "Permutation importance"]):
    ax.bar(feat_labels, scores, color=colours)
    ax.set_title(title)
    ax.set_ylabel("Importance")
    ax.set_xticklabels(feat_labels, rotation=30, ha="right")
plt.suptitle("MDI bias: high-cardinality noise inflates MDI (red bar)", y=1.02)
plt.tight_layout(); plt.show()

print("MDI rank of noise_HC (0 = highest):   ", sorted(mdi_bias, reverse=True).index(mdi_bias[5]))
print("Perm rank of noise_HC (0 = highest):  ", sorted(perm_bias, reverse=True).index(perm_bias[5]))

MDI assigns inflated importance to `noise_HC` because the tree can always find
some threshold on a continuous feature with 100 unique values that gives a small
reduction in impurity, even if the feature carries no signal. Permutation importance
is not susceptible to this because it measures the drop in test accuracy when the
feature is randomly shuffled — a pure noise feature produces no drop regardless
of its cardinality.

**In-class exercise.** The low-cardinality noise feature `noise_LC` (three distinct
integer values) is ranked lower than `noise_HC` by MDI. Explain why cardinality,
and not just whether a feature is informative, affects MDI rankings.

---
## Take-home exercises

**Exercise 1.** Using the Digits training set, carry out the cost-complexity pruning
experiment from Part D more carefully. Extract the full pruning sequence, identify
all crossover values of alpha (i.e. values where the optimal subtree changes), and
plot the number of leaves against alpha on a log scale. For three values of alpha ---
one before, one at, and one after the CV-optimal alpha --- state the number of leaves
and the test error.

**Exercise 2.** Reproduce the MDI calculation from the lecture by hand on a small
example. Build a depth-2 tree on a two-class subset of Digits (digits 3 and 8 only)
and extract the following quantities directly from the fitted tree object: the feature
used at each node (`tree_.feature`), the impurity at each node (`tree_.impurity`),
and the sample count at each node (`tree_.n_node_samples`). Compute MDI manually
for all internal nodes and verify that your result matches `feature_importances_`
(normalised to sum to 1).

**Exercise 3.** The lecture derives the variance of an average of $B$ correlated
estimators as $\rho\sigma^2 + (1-\rho)\sigma^2/B$. Verify this empirically on
the Digits dataset: fit $B = 50$ decision trees (depth 5) on 50 bootstrap resamples
of the training set. At each of 20 fixed test points, compute the variance of the
50 predicted class probabilities (for the correct class), the mean pairwise
correlation of the 50 prediction vectors, and compare $\rho\sigma^2 +
(1-\rho)\sigma^2/B$ to the empirical variance of the averaged prediction. Do the
theoretical and empirical quantities agree?

**Exercise 4.** Repeat the MDI bias experiment from Part F, but vary the cardinality
of the noise feature: use $m \in \{2, 5, 10, 50, 200\}$ distinct values. For each,
compute the MDI rank of the noise feature among all 7 features. Plot the MDI rank
against $m$ and describe the trend. At what cardinality does the noise feature
consistently rank above all five genuinely informative features?